# Phase 2: Siamese Network Fine-Tuning with Triplet Margin Loss

**Objective:** To fine-tune the pre-trained ResNet18 backbone using a Triplet architecture with Cosine Distance. This process trains the model to strictly differentiate between the unique stroke dynamics of a genuine master signature and skilled/forged signatures, enforcing an angular margin separation between genuine and forged specimens.

In [2]:
import os
import glob
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

class TripletDataset(Dataset):
    '''Dataset yielding (anchor, positive, negative) image triplets for metric learning.'''
    def __init__(self, triplets, transform=None):
        self.triplets = triplets
        self.transform = transform

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, index):
        a_path, p_path, n_path = self.triplets[index]
        
        img_a = cv2.cvtColor(cv2.imread(a_path), cv2.COLOR_BGR2RGB)
        img_p = cv2.cvtColor(cv2.imread(p_path), cv2.COLOR_BGR2RGB)
        img_n = cv2.cvtColor(cv2.imread(n_path), cv2.COLOR_BGR2RGB)

        if self.transform:
            img_a = self.transform(img_a)
            img_p = self.transform(img_p)
            img_n = self.transform(img_n)
        return img_a, img_p, img_n

print('TripletDataset class initialized successfully.')

TripletDataset class initialized successfully.


### 1. Data Augmentation Strategy

To prevent overfitting on a limited dataset of genuine signatures, we apply stochastic data augmentations. These transformations simulate real-world document scanning imperfections, such as slight rotations, brightness variations, and minor affine shifts, making the model highly robust to poor document quality.

In [3]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Data augmentation pipeline ready.")

Data augmentation pipeline ready.


### 2. Model Architecture and Triplet Optimization Setup

We initialize the ResNet18 feature extractor in training mode. To tightly align with biometric forensic metric learning, we utilize `TripletMarginWithDistanceLoss` using cosine distance $d(x, y) = 1.0 - \text{cosine\_similarity}(x, y)$. This function pushes the anchor-positive pair together while enforcing an angular distance margin of at least $0.4$ against negative/forged specimens.

In [4]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet = torch.nn.Sequential(*(list(resnet.children())[:-1]))
resnet.train()

# Triplet Margin Loss with Cosine Distance
criterion = nn.TripletMarginWithDistanceLoss(
    distance_function=lambda x, y: 1.0 - F.cosine_similarity(x, y),
    margin=0.4
)
optimizer = optim.Adam(resnet.parameters(), lr=0.0005)

os.makedirs('../models', exist_ok=True)

print('ResNet18 backbone, Triplet Margin Loss (Cosine Distance), and Optimizer initialized.')

ResNet18 backbone, Triplet Margin Loss (Cosine Distance), and Optimizer initialized.


### 3. Data Standardization (Apple-to-Apple Preprocessing)

A critical step to prevent the model from learning environmental biases (e.g., memorizing paper size rather than ink strokes). We apply the exact same adaptive thresholding and tight-cropping pipeline used on the master signatures to the external dataset. This guarantees all training inputs share identical dimensional properties.

In [5]:
os.makedirs('../data/processed/kaggle_cropped', exist_ok=True)
# Load external signers directly from extract/
kaggle_raw = sorted(glob.glob('../extract/*/original_*.jpg'))[:100]

count = 0
for path in kaggle_raw:
    img = cv2.imread(path)
    if img is None:
        continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)
        pad = 15
        x_p, y_p = max(0, x - pad), max(0, y - pad)
        w_p, h_p = min(img.shape[1] - x_p, w + (pad * 2)), min(img.shape[0] - y_p, h + (pad * 2))
        crop = img[y_p:y_p + h_p, x_p:x_p + w_p]
        cv2.imwrite(f'../data/processed/kaggle_cropped/k_{count}.jpg', crop)
        count += 1

print(f'Successfully cropped and standardized {count} external signatures.')

Successfully cropped and standardized 100 external signatures.


### 4. Triplet Generation (Anchor, Positive, Hard Negative)

We construct the training dataset by generating triplets $(Anchor, Positive, Negative)$. For Signer 1:
- **Anchor**: Genuine signature
- **Positive**: Another genuine signature from the same signer
- **Hard Negative**: Skilled forgery of Signer 1 (prioritized for stroke-level discrimination)
- **Random Negative**: External signatures from other signers

In [6]:
# Load Signer 1 genuine signatures, skilled forgeries, and cropped external signatures from extract/
asli_paths = sorted(glob.glob('../extract/001/original_1_*.jpg'))
forgery_paths = sorted(glob.glob('../extract/001_forg/forgeries_1_*.jpg'))
kaggle_paths = sorted(glob.glob('../data/processed/kaggle_cropped/*.jpg'))

triplets = []

# Generate triplets: (Anchor: Genuine, Positive: Genuine, Negative: Skilled/External)
for i in range(len(asli_paths)):
    for j in range(len(asli_paths)):
        if i == j:
            continue
        anchor_path = asli_paths[i]
        positive_path = asli_paths[j]
        
        # Hard negatives: skilled forgeries of Signer 1
        for forg_path in forgery_paths:
            triplets.append((anchor_path, positive_path, forg_path))
            
        # Random negatives: external signatures
        if kaggle_paths:
            sampled_ext = random.sample(kaggle_paths, min(3, len(kaggle_paths)))
            for ext_path in sampled_ext:
                triplets.append((anchor_path, positive_path, ext_path))

train_dataset = TripletDataset(triplets, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

print(f'Dataset generated with {len(triplets)} verification triplets.')

Dataset generated with 1170 verification triplets.


### 5. Training Loop and Weight Serialization

The execution phase of the fine-tuning process. The network iterates over the generated triplets, extracts normalized feature vectors, calculates the Triplet Margin Loss with cosine distance, and backpropagates the error to update its internal weights. The final optimized neural weights are serialized and saved for production deployment.

In [7]:
epochs = 10
resnet.train()

print('Initiating Siamese Network fine-tuning sequence with Triplet Loss...')

for epoch in range(epochs):
    epoch_loss = 0
    for anchor_img, pos_img, neg_img in train_loader:
        optimizer.zero_grad()
        
        # Extract features and apply L2 normalization
        dna_a = F.normalize(resnet(anchor_img).squeeze(), p=2, dim=-1)
        dna_p = F.normalize(resnet(pos_img).squeeze(), p=2, dim=-1)
        dna_n = F.normalize(resnet(neg_img).squeeze(), p=2, dim=-1)
        
        loss = criterion(dna_a, dna_p, dna_n)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{epochs}] | Average Triplet Loss: {avg_loss:.4f}')

torch.save(resnet.state_dict(), '../models/forensic_signature_v1.pt')
print("Model weights successfully saved to 'models' directory.")

Initiating Siamese Network fine-tuning sequence with Triplet Loss...
Epoch [1/10] | Average Triplet Loss: 0.2038
Epoch [2/10] | Average Triplet Loss: 0.0163
Epoch [3/10] | Average Triplet Loss: 0.0040
Epoch [4/10] | Average Triplet Loss: 0.0097
Epoch [5/10] | Average Triplet Loss: 0.0067
Epoch [6/10] | Average Triplet Loss: 0.0019
Epoch [7/10] | Average Triplet Loss: 0.0033
Epoch [8/10] | Average Triplet Loss: 0.0015
Epoch [9/10] | Average Triplet Loss: 0.0015
Epoch [10/10] | Average Triplet Loss: 0.0043
Model weights successfully saved to 'models' directory.
